In [1]:
# dataset.py = loadimage+labels and return tensor version of image
# for converting image into input tensor
from torch.utils.data import Dataset
import os
from PIL import Image
import torch

class SAR_Dataset(Dataset):
    def __init__(self, base_dir, transform=None):
        self.base_dir = base_dir
        self.transform = transform
        self.data = self._build_data()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]        # unpack tuple
        img = Image.open(img_path)
        label = torch.tensor(label, dtype=torch.long)
        if self.transform:
            img = self.transform(img)
        return img, label

    def _build_data(self):
        classes_to_idx = {"agri": 0, "barrenland": 1, "grassland": 2, "urban": 3}
        data = []
        for cls in classes_to_idx:
            s1_folder = os.path.join(self.base_dir, cls, "s1")
            for img in os.listdir(s1_folder):
                img_path = os.path.join(s1_folder, img)
                data.append((img_path, classes_to_idx[cls]))
        return data

In [2]:
model=SAR_Dataset("/kaggle/input/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain/v_2")
len(model)

16000

In [3]:
# test dataset.py
from torchvision import transforms
from PIL import Image

transform = transforms.Compose([
    transforms.Grayscale(),         # ensure 1 channel
    transforms.Resize((64, 64)),    # fix size
    transforms.ToTensor(),          # PIL → tensor (1, 64, 64)
])

dataset = SAR_Dataset("/kaggle/input/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain/v_2", transform=transform)

# Test 1 - how many images total
print("Total images:", len(dataset))

# Test 2 - look at one sample
img, label = dataset[0]
print("Image tensor shape:", img.shape)   # should be (1, 64, 64)
print("Label:", label)                    # should be tensor(0,1,2 or 3)
print("Min val:", img.min())              # after ToTensor → 0.0 to 1.0
print("Max val:", img.max())

img = Image.open("/kaggle/input/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain/v_2/agri/s1/ROIs1868_summer_s1_59_p100.png")
print(img.size)   # gives (width, height)

Total images: 16000
Image tensor shape: torch.Size([1, 64, 64])
Label: tensor(0)
Min val: tensor(0.0627)
Max val: tensor(0.9843)
(256, 256)


In [4]:
# model.py  model architechture

import torch.nn as nn

class SAR_model(nn.Module):

    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(   # group 1 , size(256,256)
            nn.Conv2d(1, 32 , 3 , padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # 256/2 = size(128,128)
            
            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # 128/2 =size(64,64)
            
            nn.Conv2d(64,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # 64/2 =size(32,32)
            
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # 32/2 = size(16,16)
            
            nn.Conv2d(128,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2,2) # 16/2 =size(8,8)
            
        )                                     
        self.classifier= nn.Sequential(    # group 2
            nn.Linear(128*8*8 ,128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128,4)
        )

    def forward(self,x):
        x=self.features(x)    # ( batch , hight, width, out_channel )
        x=x.view(x.size(0),-1) #(batch,features(hight*widht*out_channel))
        x=self.classifier(x) # ( batch , 4( final output))
        return x




In [5]:
# utils.py - config+transform+helper

import torch
from torchvision import transforms
from torch.utils.data import random_split,DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES=4
BATCH_SIZE=32
BASE_DIR="/kaggle/input/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain/v_2"
EPOCHS=10
LR=1e-4


def get_train_transforms():

    return transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((256,256)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.5],
            std=[0.5]) ])

def get_val_transforms():
    
    return transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((256,256)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.5],
            std=[0.5]) ])

def get_dataloader(base_dir):
    # Two instances, same data, different transforms
    train_dataset = SAR_Dataset(base_dir, transform=get_train_transforms())
    val_dataset   = SAR_Dataset(base_dir, transform=get_val_transforms())

    # Need same indices for both — so split on indices, not dataset objects
    total = len(train_dataset)
    train_size = int(0.8 * total)
    val_size   = total - train_size

    shuffled_index = torch.randperm(total)          # shuffled index list
    train_index = shuffled_index[:train_size]
    val_index   = shuffled_index[train_size:]

    # Subset with random index
    train_subset = torch.utils.data.Subset(train_dataset, train_index)
    val_subset   = torch.utils.data.Subset(val_dataset,   val_index)

    # Wrap in DataLoaders
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, val_loader


    
    


In [6]:
# model=SAR_model().to(DEVICE)
# criterion=CrossEntropyLoss()
# optimizer=Adam(model.parameter(),lr=LR)

# val_acc=0.0

# for EPOCH in range(EPOCHS):
    